# Module 00: Utility Functions

This module introduces the two shared utility functions used throughout the LLM curriculum:

- **`data.load_text(path)`** -- reads text from any file path using `pathlib.Path`
- **`plotting.heatmap(matrix, title)`** -- renders a 2D matrix as a color-coded heatmap via `matplotlib`

These are intentionally minimal helpers. Every subsequent module (tokenization, attention, transformers, etc.) imports them to load sample text and visualize weight matrices, attention patterns, and embeddings.

In this notebook we will:
1. Demonstrate each utility in isolation
2. Build progressively richer visualizations that preview the rest of the curriculum
3. Show how the same two functions adapt to many different use cases

In [ ]:
%matplotlib inline

import numpy as np
import torch
import matplotlib.pyplot as plt
import tempfile, os

from data import load_text
from plotting import heatmap

print("Imports OK")
print(f"NumPy  : {np.__version__}")
print(f"PyTorch: {torch.__version__}")

## 1. Loading Text with `load_text`

`load_text(path)` is a one-liner around `pathlib.Path.read_text()`. Its purpose is to give every module a single, consistent way to ingest raw text -- whether that text is a Shakespeare excerpt, a code snippet, or a synthetic dataset.

In [ ]:
# Create a temporary file with sample text and load it back
sample_text = (
    "The transformer architecture revolutionized natural language processing.\n"
    "Attention mechanisms allow models to focus on relevant parts of the input.\n"
    "Positional encodings give the model a sense of token order."
)

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False)
tmp.write(sample_text)
tmp.close()

loaded = load_text(tmp.name)
print(f"File path : {tmp.name}")
print(f"Characters: {len(loaded)}")
print(f"Lines     : {loaded.count(chr(10)) + 1}")
print("---")
print(loaded)

In [ ]:
# Verify round-trip fidelity -- loaded text matches what we wrote
assert loaded == sample_text, "Round-trip mismatch!"
print("Round-trip check passed: loaded text is identical to original.")

# Clean up
os.unlink(tmp.name)
print(f"Temp file removed.")

## 2. Visualizing Matrices with `heatmap`

`heatmap(matrix, title)` wraps `plt.imshow` with the `viridis` colormap and an automatic colorbar. It accepts any 2D array-like (NumPy array, PyTorch tensor, or plain list of lists).

Let's start with a simple random matrix.

In [ ]:
# Basic heatmap with a random matrix
np.random.seed(42)
random_matrix = np.random.rand(8, 8)

print(f"Matrix shape: {random_matrix.shape}")
print(f"Value range : [{random_matrix.min():.3f}, {random_matrix.max():.3f}]")
heatmap(random_matrix, title="Random 8x8 Matrix")

In [ ]:
# heatmap also works directly with PyTorch tensors
torch_matrix = torch.randn(6, 10)
print(f"Tensor shape: {torch_matrix.shape}")
print(f"dtype       : {torch_matrix.dtype}")
heatmap(torch_matrix.numpy(), title="Random PyTorch Tensor (6x10)")

## 3. Attention-Like Patterns

In later modules we will compute real attention weights. For now, let's synthesize a few common patterns to show how `heatmap` makes them immediately interpretable.

Three canonical patterns:
- **Uniform attention** -- every token attends equally to every other token
- **Diagonal (self) attention** -- each token attends mostly to itself
- **Causal (lower-triangular) attention** -- each token can only attend to earlier tokens

In [ ]:
seq_len = 8

# Uniform attention
uniform = np.ones((seq_len, seq_len)) / seq_len

# Diagonal (self) attention -- softmax of a scaled identity matrix
diag_logits = np.eye(seq_len) * 5.0
diag_exp = np.exp(diag_logits - diag_logits.max(axis=-1, keepdims=True))
diagonal = diag_exp / diag_exp.sum(axis=-1, keepdims=True)

# Causal attention -- uniform over allowed positions
causal_mask = np.tril(np.ones((seq_len, seq_len)))
causal = causal_mask / causal_mask.sum(axis=-1, keepdims=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, mat, title in zip(axes, [uniform, diagonal, causal],
                           ["Uniform", "Diagonal (Self)", "Causal (Lower-Tri)"]):
    im = ax.imshow(mat, cmap="viridis", aspect="auto", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

Each row in the matrices above sums to 1 (a valid probability distribution). The causal pattern is especially important -- it is the default attention mask in decoder-only models like GPT.

## 4. Curriculum Preview -- How These Utilities Are Used

| Module | `load_text` usage | `heatmap` usage |
|--------|-------------------|-----------------|
| 01 Tokenization | Load corpus for BPE training | Visualize token embedding matrices |
| 02 Positional Embeddings | Load sentences for encoding | Visualize sinusoidal / RoPE patterns |
| 03 Attention Basics | Load input text for attention demo | Visualize attention weight matrices |
| 04 Transformer Block | Load text for end-to-end forward pass | Visualize layer outputs |
| 05 Sampling | Load prompt text | -- |
| 06 KV Cache | Load text for cached generation | Visualize cached key/value patterns |

The same two functions adapt to every context because they operate on the universal primitives of the curriculum: **text** and **2D numerical arrays**.

## 5. Helper Visualizations

Let's build three richer visualizations that preview concepts from later modules, all powered by `heatmap`.

### 5a. Cosine Similarity Matrix

Given a set of embedding vectors, the cosine similarity matrix reveals which vectors are close in direction. This is fundamental to how attention computes relevance between tokens.

In [ ]:
# Simulate 6 token embeddings of dimension 16
torch.manual_seed(7)
embeddings = torch.randn(6, 16)

# Normalize to unit vectors
norms = embeddings.norm(dim=1, keepdim=True)
normed = embeddings / norms

# Cosine similarity = dot product of unit vectors
cos_sim = (normed @ normed.T).numpy()

print(f"Embedding shape : {embeddings.shape}")
print(f"Similarity range: [{cos_sim.min():.3f}, {cos_sim.max():.3f}]")
print(f"Diagonal values : {np.diag(cos_sim).round(3)}  (self-similarity = 1.0)")
heatmap(cos_sim, title="Cosine Similarity Between Token Embeddings")

The diagonal is always 1.0 (a vector is perfectly similar to itself). Off-diagonal values show how related pairs of embeddings are. In a trained model, semantically related tokens would show higher similarity.

### 5b. Embedding Heatmap

Visualizing the raw embedding matrix lets us see the distribution of values across dimensions. Each row is one token, each column is one embedding dimension.

In [ ]:
# Simulate a small vocabulary embedding table
vocab_labels = ["the", "cat", "sat", "on", "mat", "dog", "ran", "to"]
torch.manual_seed(12)
embed_dim = 32
embedding_table = torch.randn(len(vocab_labels), embed_dim) * 0.5

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(embedding_table.numpy(), cmap="viridis", aspect="auto")
ax.set_yticks(range(len(vocab_labels)))
ax.set_yticklabels(vocab_labels)
ax.set_xlabel("Embedding dimension")
ax.set_ylabel("Token")
ax.set_title("Embedding Table (random initialization)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(f"Table shape: {embedding_table.shape}")
print(f"Mean: {embedding_table.mean():.4f}, Std: {embedding_table.std():.4f}")

At initialization the values are roughly centered around zero with small variance. After training, specific dimensions would encode meaningful features -- syntax, semantics, frequency, etc.

### 5c. Synthetic Attention Pattern

Let's compute a realistic attention weight matrix using the scaled dot-product formula:

$$\text{Attention}(Q, K) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$$

We add a causal mask so each position can only attend to earlier positions (and itself).

In [ ]:
# Scaled dot-product attention with causal mask
torch.manual_seed(99)
seq_len = 10
d_k = 16

Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)

# Compute raw scores
scores = (Q @ K.T) / (d_k ** 0.5)

# Apply causal mask: set future positions to -inf
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
scores = scores.masked_fill(causal_mask, float("-inf"))

# Softmax to get attention weights
attn_weights = torch.softmax(scores, dim=-1)

print(f"Q shape       : {Q.shape}")
print(f"Scores shape  : {scores.shape}")
print(f"Weights shape : {attn_weights.shape}")
print(f"Row sums      : {attn_weights.sum(dim=-1).tolist()}")
heatmap(attn_weights.numpy(), title="Scaled Dot-Product Attention (Causal Masked)")

Notice the strict lower-triangular structure: the upper triangle is zero because the causal mask prevents attending to future tokens. Each row sums to 1.0. The varying brightness across allowed positions shows that some earlier tokens receive more attention than others -- this is the mechanism that lets the model learn contextual dependencies.

In [ ]:
# Compare row entropy to understand attention sharpness
# Low entropy = focused attention, high entropy = diffuse attention
log_weights = torch.log(attn_weights + 1e-9)
entropy = -(attn_weights * log_weights).sum(dim=-1)

# Maximum possible entropy for each row (uniform over allowed positions)
max_entropy = torch.log(torch.arange(1, seq_len + 1, dtype=torch.float))

fig, ax = plt.subplots(figsize=(8, 4))
positions = range(seq_len)
ax.bar(positions, entropy.numpy(), alpha=0.7, label="Actual entropy")
ax.plot(positions, max_entropy.numpy(), "r--o", markersize=5, label="Max entropy (uniform)")
ax.set_xlabel("Query position")
ax.set_ylabel("Entropy (nats)")
ax.set_title("Attention Entropy per Position")
ax.legend()
ax.set_xticks(positions)
plt.tight_layout()
plt.show()

Position 0 has zero entropy because it can only attend to itself. As positions increase, they have more tokens to distribute attention across, so the maximum possible entropy grows logarithmically. The gap between the red dashed line (maximum entropy) and the blue bars (actual entropy) indicates how focused the attention is: a large gap means the model is concentrating on a few positions rather than spreading attention uniformly.

## 6. Combining Both Utilities in a Pipeline

A typical workflow in later modules loads text, tokenizes it, computes something interesting, and visualizes the result. Here is a mini end-to-end example using both `load_text` and `heatmap`.

In [ ]:
# Write a small corpus to disk, load it, compute character-level co-occurrence
corpus = "attention is all you need\nthe quick brown fox jumps over the lazy dog"

tmp2 = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False)
tmp2.write(corpus)
tmp2.close()

text = load_text(tmp2.name)
print(f"Loaded {len(text)} characters from {tmp2.name}")

# Build character bigram co-occurrence matrix
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
n = len(chars)
cooccurrence = np.zeros((n, n))

for a, b in zip(text[:-1], text[1:]):
    cooccurrence[char_to_idx[a], char_to_idx[b]] += 1

# Normalize rows
row_sums = cooccurrence.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # avoid division by zero
cooccurrence_norm = cooccurrence / row_sums

print(f"Vocabulary size: {n} characters")
print(f"Characters     : {chars}")

# Visualize with labeled axes
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cooccurrence_norm, cmap="viridis", aspect="auto")
ax.set_xticks(range(n))
ax.set_yticks(range(n))
display_chars = ['SPC' if c == ' ' else 'NL' if c == '\n' else c for c in chars]
ax.set_xticklabels(display_chars, fontsize=8)
ax.set_yticklabels(display_chars, fontsize=8)
ax.set_xlabel("Next character")
ax.set_ylabel("Current character")
ax.set_title("Character Bigram Transition Probabilities")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

os.unlink(tmp2.name)

This pipeline -- load text, compute a matrix, visualize it -- is the pattern you will see repeated in every module. The co-occurrence matrix above is a simple language model: each row gives the probability distribution over the next character given the current one.

## Key Insights

1. **`load_text` is intentionally minimal.** It is a one-line wrapper around `pathlib.Path.read_text()`. The value is consistency: every module loads data the same way, making the curriculum easy to follow and modify.

2. **`heatmap` turns numbers into intuition.** A 10x10 matrix of floats is hard to interpret; a heatmap immediately reveals structure -- diagonals, triangles, clusters, outliers.

3. **Both utilities operate on universal primitives.** Text and 2D arrays are the fundamental data types of the entire LLM pipeline. From tokenization through attention through generation, every step either produces or consumes one of these two types.

4. **Attention patterns have characteristic shapes.** Uniform, diagonal, and causal patterns each tell a different story about how the model processes information. The `heatmap` function makes these shapes instantly recognizable.

5. **Entropy measures attention focus.** The gap between actual and maximum entropy quantifies how concentrated the attention distribution is -- a concept we will revisit in modules on attention, sampling, and generation.